# Nhiệm vụ 1.2: Phân tích Biến Giá (price_vnd, original_price_vnd, discount_rate) & Thử nghiệm Biến đổi Log

**Thành viên thực hiện**: TV3 (Exploratory Data Analysis)

**Tập dữ liệu đầu vào**: `data/processed/products_processed.jsonl`

**Mục tiêu**:
1. Tính toán các chỉ số thống kê mô tả toàn diện cho biến giá bán quan sát (`price_vnd`): Mean, Median, Std, IQR, Skewness, Kurtosis.
2. So sánh trực quan phân phối giá giữa **Laptop** và **Smartphone** qua biểu đồ Histogram, KDE và Boxplot.
3. Chứng minh toán học lợi ích của phép biến đổi Log đối với mô hình hồi quy (giảm Skewness và Kurtosis).
4. Khảo sát chính sách chiết khấu (`discount_rate`) theo 5 sàn bán lẻ.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / "scripts"))
sys.path.append("src/eda/scripts")

from eda_stats import (
    load_processed_products,
    compute_price_descriptive_stats,
    compute_price_log_comparison,
    compute_discount_stats_by_source
)
from eda_viz import (
    set_plotting_theme,
    plot_price_distribution_and_log_transform,
    plot_discount_rate_by_source
)

set_plotting_theme()
print("Môi trường phân tích sẵn sàng!")

## 1. Tải dữ liệu và Thống kê mô tả Giá bán (price_vnd)

In [ ]:
df = load_processed_products("data/processed/products_processed.jsonl")
price_stats_df = compute_price_descriptive_stats(df)
display(price_stats_df)

## 2. Thử nghiệm Biến đổi Log và Đánh giá Tính chuẩn (Normality)

In [ ]:
log_comp_df = compute_price_log_comparison(df)
display(log_comp_df)

In [ ]:
fig1_path = "src/eda/outputs/figures/fig_1_2_price_distribution_and_log_transform.png"
plot_price_distribution_and_log_transform(df, fig1_path)
print(f"Đã xuất biểu đồ: {fig1_path}")

## 3. Khảo sát Tỷ lệ Chiết khấu (discount_rate) theo Nguồn sàn

In [ ]:
discount_df = compute_discount_stats_by_source(df)
display(discount_df)

fig2_path = "src/eda/outputs/figures/fig_1_2_discount_rate_by_source.png"
plot_discount_rate_by_source(df, fig2_path)
print(f"Đã xuất biểu đồ: {fig2_path}")

## 4. Kết luận cốt lõi cho Báo cáo & Bàn giao TV5

1. **Phân phối giá gốc cực kỳ lệch phải (Heavy-tailed Right-skewed)**:
   * Toàn thị trường: Skewness = 3,47; Kurtosis = 34,48.
   * Smartphone có độ lệch cực đoan (Skewness = 7,38; Kurtosis = 170,71) do sự xuất hiện của các phiên bản siêu cao cấp (đạt đỉnh 485 triệu VNĐ).
   * Giá trị trung vị (Median) thấp hơn đáng kể so với Giá trị trung bình (Mean): Laptop có Median 26,84 triệu (Mean 32,99 triệu); Smartphone có Median 7,99 triệu (Mean 13,99 triệu).

2. **Hiệu quả vượt trội của phép biến đổi Log**:
   * Sau khi chuyển sang thang `log10(price_vnd)` hoặc `log1p(price_vnd)`:
     * Độ lệch của **Laptop** giảm từ 2,89 xuống **0,29** (xấp xỉ phân phối chuẩn hoàn hảo Gaussian).
     * Độ lệch của **Smartphone** giảm từ 7,38 xuống **-0,58**.
   * **Khuyến nghị bắt buộc cho TV5**: Bắt buộc sử dụng hàm mục tiêu log khi huấn luyện các mô hình hồi quy (OLS, Ridge, Lasso, Neural Network) để tránh sai số dự báo bị chi phối bởi các mẫu giá cực cao.

3. **Chính sách giảm giá giữa các sàn**:
   * Tỷ lệ chiết khấu trung vị của thị trường là **8,34%** (Mean: 11,50%).
   * FPT Shop có tỷ lệ hiển thị giảm giá cao nhất (99,88% sản phẩm có giảm giá) nhưng mức giảm vừa phải (trung vị 6,16%).
   * Hoàng Hà Mobile có mức chiết khấu trung vị cao nhất thị trường (**14,61%**), định vị là kênh bán lẻ cạnh tranh mạnh về giá.